# FNI-LLM Training — Final Notebook
## Cameroon Language Model

### Instructions
1. Enable GPU: Runtime → Change runtime type → T4 GPU → Save
2. Run **Cell 1** (Mount Drive)
3. Run **Cell 2** (Everything else — training + generation)
4. Run **Cell 3** (Push results to GitHub)

That's it. Three cells.


In [ ]:
# ============================================================
# CELL 1: Mount Drive + Sync Code
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/FNI_AI_LLM')

import subprocess
result = subprocess.run(['git', 'pull', 'origin', 'master'],
                        capture_output=True, text=True)
print(result.stdout)

import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')
print('Drive mounted and code synced!')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ============================================================
# CELL 2: COMPLETE TRAINING — Everything in one cell
# Imports → Config → Tokenizer → Model → Data → Train → Generate
# ============================================================

# --- IMPORTS ---
import re, os, json, time
import numpy as np
from collections import Counter
import torch
import torch.nn as nn

# --- VERIFY GPU ---
print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
else:
    print('WARNING: No GPU found. Training will be slow.')

# --- CONFIG ---
LANGUAGE   = 'english'   # Change to: 'french', 'bayangi', 'douala'
SEQ_LEN    = 64
MAX_VOCAB  = 50000
BATCH_SIZE = 128
EPOCHS     = 20
LR         = 3e-4
WARMUP     = 2
DATA_PATH  = f'data/cameroon_languages/{LANGUAGE}/processed/{LANGUAGE}_clean.txt'

print(f'\nConfig:')
print(f'  Language:   {LANGUAGE}')
print(f'  Vocab:      {MAX_VOCAB:,}')
print(f'  Seq len:    {SEQ_LEN}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Epochs:     {EPOCHS}')
print(f'  LR:         {LR}')

# --- TOKENIZER ---
class Tokenizer:
    def __init__(self, max_vocab=50000):
        self.max_vocab  = max_vocab
        self.w2i        = {}
        self.i2w        = {}
        self.vocab_size = 0

    def tokenize(self, text):
        return re.findall(r"[a-z']+|[.,!?]", text.lower())

    def build(self, corpus):
        freq  = Counter(self.tokenize(corpus))
        words = [w for w, _ in freq.most_common(self.max_vocab - 2)]
        vocab = ['<PAD>', '<UNK>'] + words
        self.w2i        = {w: i for i, w in enumerate(vocab)}
        self.i2w        = {i: w for w, i in self.w2i.items()}
        self.vocab_size = len(vocab)
        total     = sum(freq.values())
        unk_count = sum(
            c for w, c in freq.items()
            if w not in self.w2i)
        print(f'Vocab: {self.vocab_size:,} | '
              f'UNK: {unk_count/total:.2%} | '
              f'Total tokens: {total:,}')

    def encode(self, text):
        return [self.w2i.get(w, 1)
                for w in self.tokenize(text)]

    def decode(self, ids):
        return ' '.join(
            self.i2w.get(i, '')
            for i in ids
            if i > 1 and self.i2w.get(i, ''))

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({
                'w2i': self.w2i,
                'vocab_size': self.vocab_size
            }, f)

# --- MODEL ---
class FNITransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 d_ff=1024, num_layers=4,
                 max_seq_len=64, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding  = nn.Embedding(
            vocab_size, d_model, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_seq_len, d_model)
        self.drop       = nn.Dropout(dropout)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_ff, dropout=dropout,
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            enc, num_layers=num_layers)
        self.norm        = nn.LayerNorm(d_model)
        self.out         = nn.Linear(d_model, vocab_size, bias=False)
        self.out.weight  = self.embedding.weight
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        T   = x.shape[1]
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x   = self.drop(self.embedding(x) + self.pos_emb(pos))
        x   = self.transformer(x)
        return self.out(self.norm(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

# --- LOAD DATA ---
print(f'\nLoading corpus from {DATA_PATH}...')
with open(DATA_PATH, encoding='utf-8') as f:
    corpus = f.read()
print(f'Corpus: {len(corpus):,} chars | {corpus.count(chr(10)):,} lines')

tok = Tokenizer(max_vocab=MAX_VOCAB)
tok.build(corpus)

os.makedirs(f'models/{LANGUAGE}', exist_ok=True)
tok.save(f'models/{LANGUAGE}/tokenizer.json')
print('Tokenizer saved!')

print('Encoding corpus...')
all_ids = tok.encode(corpus)
print(f'Encoded tokens: {len(all_ids):,}')

print('Building training samples...')
samples = []
for i in range(0, len(all_ids) - SEQ_LEN, SEQ_LEN // 2):
    chunk = all_ids[i: i + SEQ_LEN + 1]
    if len(chunk) == SEQ_LEN + 1:
        samples.append(chunk)

np.random.seed(42)
np.random.shuffle(samples)
n       = len(samples)
train_s = samples[:int(n * 0.8)]
val_s   = samples[int(n * 0.8): int(n * 0.9)]
print(f'Train: {len(train_s):,} | Val: {len(val_s):,}')

# --- BUILD MODEL ---
model = FNITransformer(
    vocab_size  = tok.vocab_size,
    d_model     = 256,
    num_heads   = 8,
    d_ff        = 1024,
    num_layers  = 4,
    max_seq_len = SEQ_LEN,
    dropout     = 0.1
).to(DEVICE)
print(f'\nModel parameters: {model.count_params():,}')

# --- TRAINING HELPERS ---
def make_batches(samples, bs, shuffle=True):
    idx = np.arange(len(samples))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(idx) - bs, bs):
        arr = np.array([samples[j] for j in idx[i:i+bs]])
        yield arr[:, :-1], arr[:, 1:]

def lr_schedule(epoch):
    if epoch < WARMUP:
        return LR * (epoch + 1) / WARMUP
    p = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return LR * 0.5 * (1 + np.cos(np.pi * p))

# --- TRAINING LOOP ---
opt       = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)
best_loss = float('inf')
log       = []

os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print(f'\nTraining {LANGUAGE} for {EPOCHS} epochs...\n')
total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    for g in opt.param_groups:
        g['lr'] = lr_schedule(epoch)

    # Train
    model.train()
    tl = []
    t0 = time.time()
    for X_np, y_np in make_batches(train_s, BATCH_SIZE):
        X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
        y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
        opt.zero_grad()
        loss = criterion(
            model(X).reshape(-1, tok.vocab_size),
            y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl.append(loss.item())

    # Validate
    model.eval()
    vl = []
    with torch.no_grad():
        for X_np, y_np in make_batches(
                val_s, BATCH_SIZE, shuffle=False):
            X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
            y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
            loss = criterion(
                model(X).reshape(-1, tok.vocab_size),
                y.reshape(-1))
            vl.append(loss.item())

    tl_m    = float(np.mean(tl))
    vl_m    = float(np.mean(vl))
    elapsed = time.time() - t0

    if vl_m < best_loss:
        best_loss = vl_m
        torch.save({
            'model': model.state_dict(),
            'vocab': tok.w2i,
            'config': {
                'vocab_size':  tok.vocab_size,
                'd_model':     256,
                'num_heads':   8,
                'd_ff':        1024,
                'num_layers':  4,
                'max_seq_len': SEQ_LEN
            }
        }, f'models/checkpoints/{LANGUAGE}_best.pt')

    log.append({
        'epoch':      epoch,
        'train_loss': round(tl_m, 4),
        'val_loss':   round(vl_m, 4),
        'time':       round(elapsed, 2)
    })

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'train={tl_m:.4f} | '
          f'val={vl_m:.4f} | '
          f'lr={lr_schedule(epoch):.2e} | '
          f'{elapsed:.0f}s')

total_time = time.time() - total_start
print(f'\nTotal training time: {total_time/60:.1f} minutes')
print(f'Best val loss: {best_loss:.4f}')
print(f'Saved: models/checkpoints/{LANGUAGE}_best.pt')

with open(f'logs/{LANGUAGE}_final_training.json', 'w') as f:
    json.dump(log, f, indent=2)

# --- SAVE TRAINING CURVE ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

tl_vals = [e['train_loss'] for e in log]
vl_vals = [e['val_loss']   for e in log]

plt.figure(figsize=(10, 4))
plt.plot(tl_vals, label='Train Loss')
plt.plot(vl_vals, label='Val Loss')
plt.title(f'FNI-LLM Training - {LANGUAGE.capitalize()}')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
os.makedirs('docs/visualizations', exist_ok=True)
plt.savefig(f'docs/visualizations/{LANGUAGE}_final_curve.png', dpi=100)
plt.close()
print(f'Curve saved: docs/visualizations/{LANGUAGE}_final_curve.png')

# --- GENERATE TEXT ---
def generate(prompt, max_new=25, temp=0.8, top_k=40):
    model.eval()
    ids  = tok.encode(prompt)
    seen = {}
    with torch.no_grad():
        for _ in range(max_new):
            x      = torch.tensor(
                [ids[-SEQ_LEN:]], dtype=torch.long).to(DEVICE)
            logits = model(x)[0, -1].cpu().numpy().astype(float)
            logits[0] = logits[1] = -1e9
            for tid, cnt in seen.items():
                logits[tid] -= 2.0 * cnt
            logits   = logits / temp
            top_idx  = np.argsort(logits)[-top_k:]
            probs    = np.exp(logits[top_idx] - np.max(logits[top_idx]))
            probs   /= probs.sum()
            nxt      = int(np.random.choice(top_idx, p=probs))
            seen[nxt] = seen.get(nxt, 0) + 1
            ids.append(nxt)
    prompt_len = len(tok.encode(prompt))
    return tok.decode(ids[prompt_len:])

# Load best checkpoint
ckpt = torch.load(
    f'models/checkpoints/{LANGUAGE}_best.pt',
    map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print('\nLoaded best checkpoint')

prompts = [
    'cameroon is a country in central africa',
    'the people of cameroon speak',
    'douala is the largest city',
    'the official languages of cameroon are',
    'the history of cameroon began',
    'language models are trained on large',
    'artificial intelligence can help',
]

print(f'\n=== GENERATED TEXT ({LANGUAGE.upper()}) ===')
for p in prompts:
    out = generate(p, max_new=20, temp=0.8, top_k=40)
    print(f'Prompt : "{p}"')
    print(f'Output : "{out}"')
    print()

In [ ]:
# ============================================================
# CELL 3: Push Results to GitHub
# ============================================================
import os
os.chdir('/content/drive/MyDrive/FNI_AI_LLM')

!git config --global user.email 'coursdenatationcmr@gmail.com'
!git config --global user.name 'Ronald'
!git add logs/ docs/visualizations/ models/english/
!git commit -m '[Year3] English final training complete - 1M sentences'
!git push origin master
print('Results pushed to GitHub!')